# Research Module 2: Data Science and AI Evaluation

**AI-Ready Radiology Curriculum**

In this notebook you will:
1. Perform deeper data analysis using pandas (filtering, grouping, aggregation)
2. Calculate AI evaluation metrics: sensitivity, specificity, PPV, NPV
3. Build a confusion matrix
4. Explore the sensitivity/specificity trade-off by adjusting confidence thresholds
5. Create an ROC curve
6. Perform subgroup analysis to detect performance disparities
7. Load and display a radiology image

---

**Prerequisites:** Research Module 1 (Python basics, pandas, GitHub). If `df.head()` or `value_counts()` feel unfamiliar, revisit R1 first.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('Libraries loaded.')

In [ ]:
# ============================================================
# Replace YOUR-USERNAME with your GitHub username
# ============================================================
GITHUB_USERNAME = 'YOUR-USERNAME'

url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/research-starter/main/radiology_ai_findings.csv'
df = pd.read_csv(url)

print(f'Loaded {len(df)} studies')
df.head()

## Part 1: Deeper Data Analysis

In R1 you used `value_counts()` to count categories. Now we introduce more powerful pandas tools.

### Filtering

You can select rows that match a condition using bracket notation.

In [ ]:
# Filter: only chest X-rays
chest_only = df[df['modality'] == 'CR']
print(f'Chest X-rays: {len(chest_only)} studies')

# Filter: only studies the AI flagged
ai_positive = df[df['ai_flagged'] == True]
print(f'AI flagged: {len(ai_positive)} studies')

# Combine conditions with & (and) or | (or)
chest_flagged = df[(df['modality'] == 'CR') & (df['ai_flagged'] == True)]
print(f'Chest X-rays flagged by AI: {len(chest_flagged)} studies')

### Groupby and Aggregation

`groupby()` splits data into groups and applies a function to each group. This is how we compute metrics per modality, per body region, etc.

In [ ]:
# Count AI-flagged studies per modality
flags_by_mod = df.groupby('modality')['ai_flagged'].sum()
print('AI-flagged count per modality:')
print(flags_by_mod)
print()

# Mean AI confidence per modality
conf_by_mod = df.groupby('modality')['ai_confidence'].mean()
print('Mean AI confidence per modality:')
print(conf_by_mod.round(3))

---

## Part 2: AI Evaluation Metrics

When evaluating an AI diagnostic tool, we compare its predictions to a ground truth (here, the radiologist's confirmation). Every prediction falls into one of four categories:

| | Radiologist: Finding | Radiologist: Normal |
|---|---|---|
| **AI: Flagged** | True Positive (TP) | False Positive (FP) |
| **AI: Not flagged** | False Negative (FN) | True Negative (TN) |

From these four numbers we derive the core metrics:

| Metric | Formula | What it answers |
|--------|---------|------------------|
| **Sensitivity** (Recall) | TP / (TP + FN) | "Of all real findings, how many did AI catch?" |
| **Specificity** | TN / (TN + FP) | "Of all normal studies, how many did AI correctly call normal?" |
| **PPV** (Precision) | TP / (TP + FP) | "When AI flags something, how often is it real?" |
| **NPV** | TN / (TN + FN) | "When AI says normal, how often is it truly normal?" |

In [ ]:
# Calculate confusion matrix components
tp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == True)])
fp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == False)])
fn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == True)])
tn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == False)])

print('=== Confusion Matrix ===')
print(f'True Positives (TP):  {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN):  {tn}')
print(f'Total:                {tp + fp + fn + tn}')

In [ ]:
# Calculate metrics
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)

print('=== Overall AI Performance ===')
print(f'Sensitivity: {sensitivity:.1%}')
print(f'Specificity: {specificity:.1%}')
print(f'PPV:         {ppv:.1%}')
print(f'NPV:         {npv:.1%}')

### Visualize the Confusion Matrix

In [ ]:
matrix = np.array([[tp, fp], [fn, tn]])
labels = np.array([[f'TP\n{tp}', f'FP\n{fp}'], [f'FN\n{fn}', f'TN\n{tn}']])

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(matrix, cmap='Blues', aspect='auto')

for i in range(2):
    for j in range(2):
        color = 'white' if matrix[i, j] > matrix.max() / 2 else 'black'
        ax.text(j, i, labels[i, j], ha='center', va='center', fontsize=16, fontweight='bold', color=color)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Radiologist: Finding', 'Radiologist: Normal'])
ax.set_yticklabels(['AI: Flagged', 'AI: Not Flagged'])
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 3: Threshold Analysis

The AI assigns a **confidence score** (0.0 to 1.0) to each study. By default, it flags everything above some threshold. But what if we change that threshold?

- **Lower threshold**: flags more studies (higher sensitivity, lower specificity)
- **Higher threshold**: flags fewer studies (lower sensitivity, higher specificity)

This is one of the most important decisions when deploying AI clinically.

In [ ]:
thresholds = np.arange(0.10, 1.00, 0.05)
sens_list = []
spec_list = []

for thresh in thresholds:
    flagged = df['ai_confidence'] >= thresh
    t_tp = len(df[flagged & (df['radiologist_confirmed'] == True)])
    t_fp = len(df[flagged & (df['radiologist_confirmed'] == False)])
    t_fn = len(df[~flagged & (df['radiologist_confirmed'] == True)])
    t_tn = len(df[~flagged & (df['radiologist_confirmed'] == False)])
    
    sens = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
    spec = t_tn / (t_tn + t_fp) if (t_tn + t_fp) > 0 else 0
    sens_list.append(sens)
    spec_list.append(spec)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, sens_list, 'o-', color='#0EA5E9', linewidth=2, label='Sensitivity')
ax.plot(thresholds, spec_list, 's-', color='#0B1D3A', linewidth=2, label='Specificity')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('Sensitivity vs. Specificity at Different Thresholds')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ROC Curve

The **ROC curve** (Receiver Operating Characteristic) plots sensitivity vs. (1 - specificity) at every possible threshold. The **AUC** (Area Under the Curve) summarizes overall performance: 1.0 = perfect, 0.5 = random guessing.

In [ ]:
# Build ROC data
fpr_list = [1 - s for s in spec_list]

# Calculate AUC using the trapezoidal rule
# Sort by FPR for proper AUC calculation
paired = sorted(zip(fpr_list, sens_list))
fpr_sorted = [p[0] for p in paired]
tpr_sorted = [p[1] for p in paired]

auc = 0
for i in range(1, len(fpr_sorted)):
    auc += (fpr_sorted[i] - fpr_sorted[i-1]) * (tpr_sorted[i] + tpr_sorted[i-1]) / 2

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr_list, sens_list, 'o-', color='#0D9488', linewidth=2, markersize=4)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr_sorted, tpr_sorted, alpha=0.1, color='#0D9488')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title(f'ROC Curve (AUC = {auc:.3f})')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC = {auc:.3f}')

---

## Part 4: Subgroup Analysis

An AI tool might perform well overall but poorly for specific modalities, body regions, or patient subgroups. Subgroup analysis is essential for detecting these disparities.

In [ ]:
print('=== AI Sensitivity and Specificity by Modality ===')
print(f'{"Modality":<10} {"Sens":<10} {"Spec":<10} {"TP":<5} {"FP":<5} {"FN":<5} {"TN":<5}')
print('-' * 50)

for mod in sorted(df['modality'].unique()):
    s = df[df['modality'] == mod]
    m_tp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==True)])
    m_fp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==False)])
    m_fn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==True)])
    m_tn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==False)])
    
    m_sens = m_tp / (m_tp + m_fn) if (m_tp + m_fn) > 0 else 0
    m_spec = m_tn / (m_tn + m_fp) if (m_tn + m_fp) > 0 else 0
    
    print(f'{mod:<10} {m_sens:<10.1%} {m_spec:<10.1%} {m_tp:<5} {m_fp:<5} {m_fn:<5} {m_tn:<5}')

In [ ]:
# Visualize: sensitivity by modality
modalities_sorted = sorted(df['modality'].unique())
sens_by_mod = []
spec_by_mod = []

for mod in modalities_sorted:
    s = df[df['modality'] == mod]
    m_tp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==True)])
    m_fn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==True)])
    m_fp = len(s[(s['ai_flagged']==True) & (s['radiologist_confirmed']==False)])
    m_tn = len(s[(s['ai_flagged']==False) & (s['radiologist_confirmed']==False)])
    sens_by_mod.append(m_tp / (m_tp + m_fn) if (m_tp + m_fn) > 0 else 0)
    spec_by_mod.append(m_tn / (m_tn + m_fp) if (m_tn + m_fp) > 0 else 0)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(modalities_sorted))
width = 0.35
ax.bar([i - width/2 for i in x], sens_by_mod, width, label='Sensitivity', color='#0EA5E9')
ax.bar([i + width/2 for i in x], spec_by_mod, width, label='Specificity', color='#0B1D3A')
ax.set_xticks(x)
ax.set_xticklabels(modalities_sorted)
ax.set_ylabel('Metric Value')
ax.set_title('AI Sensitivity and Specificity by Modality')
ax.set_ylim(0, 1.15)
ax.legend()

for i, (se, sp) in enumerate(zip(sens_by_mod, spec_by_mod)):
    ax.text(i - width/2, se + 0.02, f'{se:.0%}', ha='center', fontsize=9, fontweight='bold')
    ax.text(i + width/2, sp + 0.02, f'{sp:.0%}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 5: Loading a Radiology Image

As a preview of Research Module 3, let's load and display an image using Python. We will use PIL (Python Imaging Library) and matplotlib.

In [ ]:
from PIL import Image
import urllib.request
from io import BytesIO

# Load a sample chest X-ray from a public source
# This is a placeholder URL - in R3 you will work with real DICOM images
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_Xray_PA_3-8-2010.png/250px-Chest_Xray_PA_3-8-2010.png'

try:
    response = urllib.request.urlopen(sample_url)
    img = Image.open(BytesIO(response.read()))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Inverted
    img_array = np.array(img)
    axes[1].imshow(255 - img_array, cmap='gray')
    axes[1].set_title('Inverted')
    axes[1].axis('off')
    
    # High contrast
    axes[2].imshow(img_array, cmap='gray', vmin=50, vmax=200)
    axes[2].set_title('Enhanced Contrast')
    axes[2].axis('off')
    
    plt.suptitle('Image Processing Preview', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f'Image size: {img.size}')
    print(f'Array shape: {img_array.shape}')
except Exception as e:
    print(f'Could not load image: {e}')
    print('This is just a preview. Full image processing is covered in Research Module 3.')

---

## Your Turn: Independent Analysis

Complete **all three tasks** below.

### Task 1: Body Region Subgroup Analysis

Pick **one body region** from the dataset and calculate sensitivity, specificity, PPV, and NPV for that region. Create a visualization.

In [ ]:
# ============================================================
# TASK 1: Pick a body region and calculate all four metrics
# ============================================================
MY_REGION = 'Chest'  # <-- Change this

region_df = df[df['body_region'] == MY_REGION]
print(f'Analyzing: {MY_REGION} ({len(region_df)} studies)')

# Your code: calculate TP, FP, FN, TN for this region
# Then calculate sensitivity, specificity, PPV, NPV
# Then create a chart


### Task 2: Threshold Recommendation

Based on the threshold analysis in Part 3, recommend a confidence threshold for clinical use. Print your recommended threshold and the sensitivity/specificity at that threshold.

In [ ]:
# ============================================================
# TASK 2: Pick your recommended threshold and justify it
# ============================================================
MY_THRESHOLD = 0.70  # <-- Change this

# Calculate metrics at your threshold
flagged_at_thresh = df['ai_confidence'] >= MY_THRESHOLD
t_tp = len(df[flagged_at_thresh & (df['radiologist_confirmed'] == True)])
t_fp = len(df[flagged_at_thresh & (df['radiologist_confirmed'] == False)])
t_fn = len(df[~flagged_at_thresh & (df['radiologist_confirmed'] == True)])
t_tn = len(df[~flagged_at_thresh & (df['radiologist_confirmed'] == False)])

t_sens = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
t_spec = t_tn / (t_tn + t_fp) if (t_tn + t_fp) > 0 else 0

print(f'Recommended threshold: {MY_THRESHOLD}')
print(f'Sensitivity at {MY_THRESHOLD}: {t_sens:.1%}')
print(f'Specificity at {MY_THRESHOLD}: {t_spec:.1%}')
print(f'Studies flagged: {flagged_at_thresh.sum()} of {len(df)}')

### Task 3: Written Interpretation

In the markdown cell below, write 3-5 sentences answering:

1. Which modality had the **highest** sensitivity? Which had the **lowest**? Why might that be?
2. What is the trade-off you observed when changing the confidence threshold?
3. How does this connect to the **never-skilling** and **mis-skilling** risks from Lesson 1? (Consider: if a trainee only sees AI-flagged cases, what metrics would they never learn to evaluate?)

**Your interpretation:**

*Replace this text with your 3-5 sentence analysis. Reference specific numbers from your outputs above.*



---

## Save Your Work

Run the completion record cell, then save to GitHub:
1. **File > Save a copy in GitHub**
2. Select your `research-starter` fork
3. Commit message: `Completed Research Module 2`

In [ ]:
from datetime import datetime

print('=' * 50)
print('RESEARCH MODULE 2 — COMPLETION RECORD')
print('=' * 50)
print(f'GitHub Username:    {GITHUB_USERNAME}')
print(f'Completed:          {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Region Analyzed:    {MY_REGION}')
print(f'Threshold Chosen:   {MY_THRESHOLD}')
print(f'Overall Sensitivity: {sensitivity:.1%}')
print(f'Overall Specificity: {specificity:.1%}')
print(f'AUC:                {auc:.3f}')
print('=' * 50)
print('Save this notebook to GitHub to submit your work.')